# Módulo 4 · Clase 1 — Introducción a bases de datos

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Bloque | Qué aprenderás |
|---|--------|----------------|
| 1 | ¿Qué es una base de datos? | Por qué no basta con archivos CSV |
| 2 | El modelo relacional | Tablas, filas, columnas |
| 3 | Claves primarias y foráneas | El pegamento que conecta las tablas |
| 4 | SQL — el lenguaje | DDL vs DML |
| 5 | `CREATE TABLE` | Definir la estructura |
| 6 | `INSERT INTO` | Añadir datos |
| 7 | `UPDATE` y `DELETE` | Modificar y eliminar |
| 8 | Tipos de datos en SQL | TEXT, INTEGER, REAL, BLOB, NULL |
| 9 | Práctica | Crear y poblar la base de datos de una tienda |

> 💡 **Idea central:** Una base de datos relacional es una colección organizada de tablas relacionadas entre sí. SQL es el lenguaje universal para hablar con ellas.

---
## 1. ¿Por qué no basta con archivos CSV?

```
PROBLEMA CON CSV:

clientes.csv:
id,nombre,email,ciudad
1,Ana,ana@mail.com,Madrid
2,Carlos,carlos@mail.com,Barcelona

pedidos.csv:
id,cliente_nombre,cliente_email,producto,precio
101,Ana,ana@mail.com,Teclado,45.00
102,Ana,ana@mail.com,Ratón,25.00
103,Carlos,carlos@mail.com,Monitor,350.00

PROBLEMAS:
❌ Duplicación: nombre y email de Ana repetidos en cada pedido
❌ Inconsistencia: si Ana cambia su email, hay que actualizarlo en dos sitios
❌ Sin integridad: un pedido puede referenciar a un cliente inexistente
❌ Sin concurrencia: dos personas editando el mismo CSV a la vez → corrupción
❌ Sin índices: buscar un pedido recorre todo el archivo línea a línea
```

**Una base de datos relacional resuelve todos estos problemas:**
- Almacena cada dato una sola vez.
- Mantiene la integridad (no puedes crear un pedido para un cliente que no existe).
- Soporta múltiples usuarios simultáneos.
- Tiene índices para búsquedas rápidas en millones de registros.

---
## 2. El modelo relacional

El modelo relacional organiza los datos en **tablas** (también llamadas *relaciones*).

```
TABLA: clientes
┌────┬──────────┬─────────────────┬───────────┐
│ id │ nombre   │ email           │ ciudad    │  ← columnas (atributos)
├────┼──────────┼─────────────────┼───────────┤
│  1 │ Ana      │ ana@mail.com    │ Madrid    │  ← fila (registro / tupla)
│  2 │ Carlos   │ carlos@mail.com │ Barcelona │
│  3 │ Lucía    │ lucia@mail.com  │ Sevilla   │
└────┴──────────┴─────────────────┴───────────┘
  ↑
  clave primaria
```

**Conceptos fundamentales:**

| Concepto | También se llama | Descripción |
|----------|-----------------|-------------|
| Tabla | Relación, entidad | Colección de datos del mismo tipo |
| Fila | Registro, tupla | Un elemento individual de la tabla |
| Columna | Atributo, campo | Una propiedad de cada elemento |
| Celda | Valor | El dato concreto |
| Esquema | Schema | La definición de la estructura (no los datos) |

---
## 3. Claves primarias y foráneas

### Clave primaria (`PRIMARY KEY`)

Identifica **de forma única** cada fila de una tabla.
- Nunca puede ser `NULL`.
- No puede repetirse.
- Suele ser un número entero autoincremental.

### Clave foránea (`FOREIGN KEY`)

Una columna que hace referencia a la clave primaria de otra tabla. Es el mecanismo que **conecta tablas** entre sí.

```
clientes                    pedidos
┌────┬────────┐             ┌─────┬─────────────┬─────────┐
│ id │ nombre │             │ id  │ cliente_id  │ total   │
├────┼────────┤             ├─────┼─────────────┼─────────┤
│  1 │ Ana    │◄────────────│ 101 │      1      │  45.00  │
│  2 │ Carlos │◄──────┐     │ 102 │      1      │  25.00  │
│  3 │ Lucía  │       └─────│ 103 │      2      │ 350.00  │
└────┴────────┘             └─────┴─────────────┴─────────┘
  ↑ PK                               ↑ FK
```

**Integridad referencial:** la FK garantiza que `cliente_id = 5` no puede existir en pedidos si no hay ningún cliente con `id = 5`.

---
## 4. SQL — el lenguaje

SQL (*Structured Query Language*) se divide en dos grandes categorías:

```
DDL — Data Definition Language    DML — Data Manipulation Language
──────────────────────────────    ────────────────────────────────
Define la ESTRUCTURA               Manipula los DATOS

CREATE TABLE  → crear tabla        SELECT → consultar
ALTER TABLE   → modificar tabla    INSERT → insertar
DROP TABLE    → eliminar tabla     UPDATE → modificar
                                   DELETE → eliminar
```

**SQLite** es el motor de base de datos que usaremos durante el módulo. Viene incluido en Python — no necesitas instalar nada. Guarda toda la base de datos en un único archivo `.db`.

```
MOTORES SQL más comunes:
SQLite   → archivo local, perfecto para desarrollo y apps pequeñas
PostgreSQL → open source, producción, muy potente
MySQL    → muy extendido en aplicaciones web
SQL Server → ecosistema Microsoft
```

In [ ]:
import sqlite3
from pathlib import Path

# Conectar (crea el archivo si no existe)
DB_PATH = Path.home() / "tienda.db"
conn = sqlite3.connect(DB_PATH)

# El cursor es el objeto que ejecuta las sentencias SQL
cur = conn.cursor()

# Activar claves foráneas en SQLite (están desactivadas por defecto)
cur.execute("PRAGMA foreign_keys = ON")

print(f"Conexión establecida: {DB_PATH}")
print(f"Versión de SQLite: {sqlite3.sqlite_version}")

---
## 5. `CREATE TABLE` — Definir la estructura

```sql
CREATE TABLE nombre_tabla (
    columna1  TIPO  RESTRICCIONES,
    columna2  TIPO  RESTRICCIONES,
    ...
);
```

**Tipos de datos en SQLite:**

| Tipo | Uso | Ejemplo |
|------|-----|---------|
| `INTEGER` | Números enteros | id, cantidad, año |
| `REAL` | Números decimales | precio, temperatura |
| `TEXT` | Texto | nombre, email, descripción |
| `BLOB` | Datos binarios | imágenes, archivos |
| `NULL` | Valor ausente | (cualquier columna puede ser NULL) |

**Restricciones comunes:**

| Restricción | Significado |
|-------------|-------------|
| `PRIMARY KEY` | Clave primaria |
| `AUTOINCREMENT` | Se incrementa solo |
| `NOT NULL` | No puede ser vacío |
| `UNIQUE` | Sin duplicados |
| `DEFAULT valor` | Valor por defecto |
| `CHECK (condición)` | Validar un valor |
| `FOREIGN KEY` | Clave foránea |

In [ ]:
# Borrar tablas si existen (para poder re-ejecutar la celda)
cur.executescript("""
    DROP TABLE IF EXISTS pedido_productos;
    DROP TABLE IF EXISTS pedidos;
    DROP TABLE IF EXISTS productos;
    DROP TABLE IF EXISTS clientes;
    DROP TABLE IF EXISTS categorias;
""")

# Tabla: categorias
cur.execute("""
    CREATE TABLE categorias (
        id        INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre    TEXT    NOT NULL UNIQUE,
        descripcion TEXT
    )
""")

# Tabla: clientes
cur.execute("""
    CREATE TABLE clientes (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre      TEXT    NOT NULL,
        email       TEXT    NOT NULL UNIQUE,
        ciudad      TEXT    DEFAULT 'Desconocida',
        fecha_alta  TEXT    DEFAULT (date('now'))
    )
""")

# Tabla: productos
cur.execute("""
    CREATE TABLE productos (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre       TEXT    NOT NULL,
        precio       REAL    NOT NULL CHECK(precio >= 0),
        stock        INTEGER NOT NULL DEFAULT 0 CHECK(stock >= 0),
        categoria_id INTEGER NOT NULL,
        FOREIGN KEY (categoria_id) REFERENCES categorias(id)
    )
""")

# Tabla: pedidos
cur.execute("""
    CREATE TABLE pedidos (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        cliente_id  INTEGER NOT NULL,
        fecha       TEXT    NOT NULL DEFAULT (date('now')),
        estado      TEXT    NOT NULL DEFAULT 'pendiente'
                            CHECK(estado IN ('pendiente','enviado','entregado','cancelado')),
        FOREIGN KEY (cliente_id) REFERENCES clientes(id)
    )
""")

# Tabla intermedia: pedido_productos (relación muchos a muchos)
cur.execute("""
    CREATE TABLE pedido_productos (
        pedido_id   INTEGER NOT NULL,
        producto_id INTEGER NOT NULL,
        cantidad    INTEGER NOT NULL CHECK(cantidad > 0),
        precio_ud   REAL    NOT NULL,
        PRIMARY KEY (pedido_id, producto_id),
        FOREIGN KEY (pedido_id)   REFERENCES pedidos(id),
        FOREIGN KEY (producto_id) REFERENCES productos(id)
    )
""")

conn.commit()
print("Tablas creadas correctamente")

# Verificar qué tablas existen
tablas = cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
print("Tablas en la base de datos:", [t[0] for t in tablas])

---
## 6. `INSERT INTO` — Añadir datos

```sql
-- Insertar una fila especificando columnas
INSERT INTO tabla (col1, col2, col3)
VALUES (val1, val2, val3);

-- Insertar varias filas a la vez
INSERT INTO tabla (col1, col2)
VALUES (v1, v2),
       (v3, v4),
       (v5, v6);

-- Sin especificar columnas (peligroso: depende del orden)
INSERT INTO tabla VALUES (val1, val2, val3);
```

> Siempre especifica las columnas — hace el código más claro y resistente a cambios en el esquema.

In [ ]:
# Insertar categorías
cur.executemany(
    "INSERT INTO categorias (nombre, descripcion) VALUES (?, ?)",
    [
        ("Electrónica",  "Dispositivos y accesorios electrónicos"),
        ("Informática",  "Equipos y periféricos informáticos"),
        ("Hogar",        "Artículos para el hogar"),
        ("Oficina",      "Material de oficina y papelería"),
    ]
)

# Insertar clientes
cur.executemany(
    "INSERT INTO clientes (nombre, email, ciudad) VALUES (?, ?, ?)",
    [
        ("Ana García",     "ana@mail.com",     "Madrid"),
        ("Carlos López",   "carlos@mail.com",  "Barcelona"),
        ("Lucía Martín",   "lucia@mail.com",   "Sevilla"),
        ("Pedro Sánchez",  "pedro@mail.com",   "Valencia"),
        ("Marta Jiménez",  "marta@mail.com",   "Madrid"),
    ]
)

# Insertar productos
cur.executemany(
    "INSERT INTO productos (nombre, precio, stock, categoria_id) VALUES (?, ?, ?, ?)",
    [
        ("Smartphone X12",   599.99, 50, 1),
        ("Auriculares BT",    89.99, 120, 1),
        ("Cargador rápido",   29.99, 200, 1),
        ("Portátil Pro 15",  1199.00, 30, 2),
        ("Teclado mecánico",  85.00, 75, 2),
        ("Ratón inalámbrico", 45.00, 90, 2),
        ("Monitor 27\" 4K",   450.00, 25, 2),
        ("Lámpara escritorio", 35.00, 60, 3),
        ("Silla ergonómica",  299.00, 15, 3),
        ("Bolígrafos 10u",     4.99, 500, 4),
        ("Agenda 2025",        12.99, 150, 4),
        ("Post-it pack",        8.50, 300, 4),
    ]
)

# Insertar pedidos
cur.executemany(
    "INSERT INTO pedidos (cliente_id, fecha, estado) VALUES (?, ?, ?)",
    [
        (1, "2025-01-10", "entregado"),
        (1, "2025-02-15", "enviado"),
        (2, "2025-01-20", "entregado"),
        (3, "2025-02-01", "entregado"),
        (3, "2025-03-05", "pendiente"),
        (4, "2025-03-10", "enviado"),
        (5, "2025-03-12", "pendiente"),
    ]
)

# Insertar líneas de pedido
cur.executemany(
    "INSERT INTO pedido_productos (pedido_id, producto_id, cantidad, precio_ud) VALUES (?, ?, ?, ?)",
    [
        (1, 1, 1, 599.99),  # pedido 1: smartphone
        (1, 3, 2,  29.99),  # pedido 1: 2 cargadores
        (2, 2, 1,  89.99),  # pedido 2: auriculares
        (2, 5, 1,  85.00),  # pedido 2: teclado
        (3, 4, 1, 1199.00), # pedido 3: portátil
        (3, 6, 1,  45.00),  # pedido 3: ratón
        (4, 8, 2,  35.00),  # pedido 4: 2 lámparas
        (5, 9, 1, 299.00),  # pedido 5: silla
        (5,10, 5,   4.99),  # pedido 5: 5 packs bolígrafos
        (6, 7, 1, 450.00),  # pedido 6: monitor
        (7,11, 2,  12.99),  # pedido 7: 2 agendas
        (7,12, 3,   8.50),  # pedido 7: 3 post-it
    ]
)

conn.commit()
print("Datos insertados")
print(f"  Categorías: {cur.execute('SELECT COUNT(*) FROM categorias').fetchone()[0]}")
print(f"  Clientes:   {cur.execute('SELECT COUNT(*) FROM clientes').fetchone()[0]}")
print(f"  Productos:  {cur.execute('SELECT COUNT(*) FROM productos').fetchone()[0]}")
print(f"  Pedidos:    {cur.execute('SELECT COUNT(*) FROM pedidos').fetchone()[0]}")

---
## 7. `UPDATE` y `DELETE` — Modificar y eliminar

```sql
-- Actualizar filas
UPDATE tabla
SET columna1 = valor1,
    columna2 = valor2
WHERE condición;

-- Eliminar filas
DELETE FROM tabla
WHERE condición;
```

⚠️ **SIEMPRE incluye `WHERE`** en `UPDATE` y `DELETE`. Sin él, afecta a **todas** las filas de la tabla.

```sql
UPDATE productos SET precio = 0;     -- ❌ pone todos los precios a 0
DELETE FROM clientes;                -- ❌ borra todos los clientes

UPDATE productos SET precio = 0 WHERE id = 5;   -- ✅
DELETE FROM clientes WHERE id = 3;              -- ✅
```

In [ ]:
# UPDATE: subir el precio del smartphone un 10%
cur.execute("SELECT nombre, precio FROM productos WHERE id = 1")
print("Antes del UPDATE:", cur.fetchone())

cur.execute("""
    UPDATE productos
    SET precio = ROUND(precio * 1.10, 2)
    WHERE id = 1
""")

cur.execute("SELECT nombre, precio FROM productos WHERE id = 1")
print("Después del UPDATE:", cur.fetchone())

# UPDATE: cambiar ciudad de Ana
cur.execute("UPDATE clientes SET ciudad = 'Málaga' WHERE email = 'ana@mail.com'")
cur.execute("SELECT nombre, ciudad FROM clientes WHERE email = 'ana@mail.com'")
print("\nCiudad de Ana:", cur.fetchone())

conn.commit()

In [ ]:
# Insertar un registro de prueba para luego borrarlo
cur.execute("INSERT INTO clientes (nombre, email, ciudad) VALUES ('Prueba', 'prueba@test.com', 'Test')")
id_prueba = cur.lastrowid
print(f"Cliente de prueba insertado con id={id_prueba}")

# Verificar
cur.execute("SELECT COUNT(*) FROM clientes")
print(f"Total clientes antes del DELETE: {cur.fetchone()[0]}")

# DELETE
cur.execute("DELETE FROM clientes WHERE id = ?", (id_prueba,))
conn.commit()

cur.execute("SELECT COUNT(*) FROM clientes")
print(f"Total clientes después del DELETE: {cur.fetchone()[0]}")

---
## Primer vistazo a los datos con `SELECT`

Aún no hemos estudiado `SELECT` en profundidad — eso es la Clase 2. Pero lo usamos ya para ver lo que tenemos.

In [ ]:
import pandas as pd

def tabla(sql, con=conn):
    return pd.read_sql_query(sql, con)

print("=== CATEGORÍAS ===")
display(tabla("SELECT * FROM categorias"))

print("\n=== CLIENTES ===")
display(tabla("SELECT * FROM clientes"))

print("\n=== PRODUCTOS ===")
display(tabla("SELECT * FROM productos ORDER BY categoria_id, nombre"))

---
## 9. Práctica — Crear y poblar una base de datos

---

### Ejercicio 1 — Crear un esquema

**Enunciado:** Crea una base de datos `biblioteca2.db` con el siguiente esquema:

- **`autores`**: id (PK), nombre (NOT NULL), nacionalidad, anio_nacimiento
- **`libros`**: id (PK), titulo (NOT NULL), isbn (UNIQUE), precio (≥ 0), paginas, autor_id (FK)
- **`socios`**: id (PK), nombre (NOT NULL), email (UNIQUE), fecha_alta (DEFAULT hoy)
- **`prestamos`**: id (PK), socio_id (FK), libro_id (FK), fecha_prestamo, fecha_devolucion (puede ser NULL), devuelto (INTEGER DEFAULT 0)

Después, verifica que las cuatro tablas se han creado correctamente.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import sqlite3
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import sqlite3
from pathlib import Path

conn2 = sqlite3.connect(Path.home() / "biblioteca2.db")
cur2 = conn2.cursor()
cur2.execute("PRAGMA foreign_keys = ON")

cur2.executescript("""
    DROP TABLE IF EXISTS prestamos;
    DROP TABLE IF EXISTS libros;
    DROP TABLE IF EXISTS socios;
    DROP TABLE IF EXISTS autores;

    CREATE TABLE autores (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre          TEXT    NOT NULL,
        nacionalidad    TEXT,
        anio_nacimiento INTEGER
    );

    CREATE TABLE libros (
        id        INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo    TEXT    NOT NULL,
        isbn      TEXT    UNIQUE,
        precio    REAL    CHECK(precio >= 0),
        paginas   INTEGER,
        autor_id  INTEGER NOT NULL,
        FOREIGN KEY (autor_id) REFERENCES autores(id)
    );

    CREATE TABLE socios (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre     TEXT    NOT NULL,
        email      TEXT    NOT NULL UNIQUE,
        fecha_alta TEXT    DEFAULT (date('now'))
    );

    CREATE TABLE prestamos (
        id                INTEGER PRIMARY KEY AUTOINCREMENT,
        socio_id          INTEGER NOT NULL,
        libro_id          INTEGER NOT NULL,
        fecha_prestamo    TEXT    NOT NULL DEFAULT (date('now')),
        fecha_devolucion  TEXT,
        devuelto          INTEGER DEFAULT 0,
        FOREIGN KEY (socio_id) REFERENCES socios(id),
        FOREIGN KEY (libro_id) REFERENCES libros(id)
    );
""")
conn2.commit()

tablas = cur2.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()
print("Tablas creadas:", [t[0] for t in tablas])
```
</details>

---
### Ejercicio 2 — Poblar la base de datos

**Enunciado:** Inserta datos en `biblioteca2.db`:

- Al menos cuatro autores con diferentes nacionalidades.
- Al menos ocho libros distribuidos entre los autores.
- Al menos cinco socios.
- Al menos seis préstamos: algunos devueltos, otros activos.

Después, muestra el recuento de registros en cada tabla.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


---
### Ejercicio 3 — UPDATE en cascada

**Enunciado:** En la base de datos `tienda.db`:

1. Aplica un descuento del 15% a todos los productos de la categoría «Oficina».
2. Cambia el estado de todos los pedidos «enviado» a «entregado».
3. Aumenta el stock de todos los productos con stock < 30 en 50 unidades.
4. Verifica cada cambio con un `SELECT`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import pandas as pd


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import sqlite3, pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path.home() / "tienda.db")
cur  = conn.cursor()

# 1. Descuento 15% en Oficina
cur.execute("""
    UPDATE productos
    SET precio = ROUND(precio * 0.85, 2)
    WHERE categoria_id = (SELECT id FROM categorias WHERE nombre = 'Oficina')
""")
print("Productos Oficina tras descuento:")
display(pd.read_sql_query("""
    SELECT p.nombre, p.precio
    FROM productos p
    JOIN categorias c ON p.categoria_id = c.id
    WHERE c.nombre = 'Oficina'
""", conn))

# 2. Enviado → entregado
cur.execute("UPDATE pedidos SET estado = 'entregado' WHERE estado = 'enviado'")
print("\nEstados de pedidos:")
display(pd.read_sql_query("SELECT estado, COUNT(*) AS total FROM pedidos GROUP BY estado", conn))

# 3. Stock bajo → +50
cur.execute("UPDATE productos SET stock = stock + 50 WHERE stock < 30")
print("\nProductos con stock previamente bajo:")
display(pd.read_sql_query("SELECT nombre, stock FROM productos ORDER BY stock", conn))

conn.commit()
```
</details>

---
### Ejercicio 4 — Integridad referencial

**Enunciado:** Verifica que las restricciones funcionan correctamente:

1. Intenta insertar un producto con `categoria_id = 99` (inexistente) — debe fallar.
2. Intenta insertar un cliente con un email ya existente — debe fallar.
3. Intenta insertar un producto con `precio = -10` — debe fallar.
4. Captura las excepciones con `try/except` y muestra el mensaje de error.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import sqlite3
from pathlib import Path

conn = sqlite3.connect(Path.home() / "tienda.db")
cur  = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON")

pruebas = [
    (
        "FK violada (categoria_id=99)",
        "INSERT INTO productos (nombre, precio, stock, categoria_id) VALUES ('X', 10, 1, 99)"
    ),
    (
        "UNIQUE violado (email duplicado)",
        "INSERT INTO clientes (nombre, email) VALUES ('Otro', 'ana@mail.com')"
    ),
    (
        "CHECK violado (precio negativo)",
        "INSERT INTO productos (nombre, precio, stock, categoria_id) VALUES ('Y', -5, 1, 1)"
    ),
]

for descripcion, sql in pruebas:
    try:
        cur.execute(sql)
        print(f"❌ {descripcion}: NO se lanzó error (algo está mal)")
    except sqlite3.IntegrityError as e:
        print(f"✅ {descripcion}")
        print(f"   Error: {e}")
    conn.rollback()
```
</details>

---
### Ejercicio 5 (reto) — Función `poblar_bd`

**Enunciado:** Escribe la función Python `poblar_bd(conn, n_clientes, n_productos)` que genere datos ficticios y los inserte:

- `n_clientes` clientes con nombres, emails únicos y ciudades aleatorias.
- `n_productos` productos con nombres, precios y stocks aleatorios distribuidos entre categorías existentes.
- Un pedido por cada cliente, cada uno con entre dos y cuatro productos aleatorios.
- Devuelve un dict con los recuentos insertados.

Pruébala con `poblar_bd(conn, 20, 30)`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import sqlite3, random
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import sqlite3, random
from pathlib import Path

NOMBRES   = ["Alba","Bea","Clara","Diana","Eva","Fátima","Gloria",
              "Héctor","Iván","Jaime","Kike","Luis","Marcos","Nacho",
              "Óscar","Pablo","Quique","Rosa","Sara","Tomás"]
CIUDADES  = ["Madrid","Barcelona","Valencia","Sevilla","Bilbao",
              "Zaragoza","Málaga","Murcia","Alicante","Valladolid"]
ADJETIVOS = ["Pro","Ultra","Lite","Plus","Max","Mini","Smart","Eco"]
TIPOS     = ["Kit","Pack","Set","Device","Tool","Gadget","System"]

def poblar_bd(conn, n_clientes=20, n_productos=30):
    cur = conn.cursor()

    # Obtener IDs de categorías existentes
    cat_ids = [r[0] for r in cur.execute("SELECT id FROM categorias").fetchall()]
    if not cat_ids:
        raise ValueError("No hay categorías — crea las tablas primero")

    # Insertar clientes
    clientes_insertados = 0
    usados = set()
    intentos = 0
    while clientes_insertados < n_clientes and intentos < n_clientes * 3:
        intentos += 1
        nombre = random.choice(NOMBRES)
        tag = random.randint(100, 9999)
        email = f"{nombre.lower()}{tag}@demo.com"
        if email in usados: continue
        usados.add(email)
        ciudad = random.choice(CIUDADES)
        try:
            cur.execute("INSERT INTO clientes (nombre, email, ciudad) VALUES (?,?,?)",
                        (nombre, email, ciudad))
            clientes_insertados += 1
        except sqlite3.IntegrityError:
            pass

    # Insertar productos
    for _ in range(n_productos):
        nombre = f"{random.choice(ADJETIVOS)} {random.choice(TIPOS)} {random.randint(1,99)}"
        precio = round(random.uniform(5, 999), 2)
        stock  = random.randint(0, 500)
        cat    = random.choice(cat_ids)
        cur.execute("INSERT INTO productos (nombre, precio, stock, categoria_id) VALUES (?,?,?,?)",
                    (nombre, precio, stock, cat))

    # Pedidos
    prod_ids = [r[0] for r in cur.execute("SELECT id FROM productos").fetchall()]
    cli_ids  = [r[0] for r in cur.execute("SELECT id FROM clientes").fetchall()]
    for cid in cli_ids[-n_clientes:]:
        cur.execute("INSERT INTO pedidos (cliente_id) VALUES (?)", (cid,))
        pid = cur.lastrowid
        for prod in random.sample(prod_ids, min(random.randint(2, 4), len(prod_ids))):
            precio = cur.execute("SELECT precio FROM productos WHERE id=?", (prod,)).fetchone()[0]
            try:
                cur.execute("INSERT INTO pedido_productos VALUES (?,?,?,?)",
                            (pid, prod, random.randint(1, 3), precio))
            except sqlite3.IntegrityError:
                pass

    conn.commit()
    return {
        "clientes":  clientes_insertados,
        "productos": n_productos,
        "pedidos":   n_clientes,
    }


conn = sqlite3.connect(Path.home() / "tienda.db")
resultado = poblar_bd(conn, 20, 30)
print("Insertados:", resultado)
```
</details>

---
## Resumen de la clase

```sql
-- ── DDL: definir estructura ────────────────────────────────
CREATE TABLE tabla (
    id    INTEGER PRIMARY KEY AUTOINCREMENT,
    col   TEXT    NOT NULL UNIQUE,
    fk_id INTEGER REFERENCES otra_tabla(id)
);

DROP TABLE IF EXISTS tabla;

-- ── DML: insertar ──────────────────────────────────────────
INSERT INTO tabla (col1, col2) VALUES (v1, v2);

INSERT INTO tabla (col1, col2)
VALUES (v1, v2), (v3, v4);       -- múltiples filas

-- ── DML: modificar ─────────────────────────────────────────
UPDATE tabla
SET col = valor
WHERE condición;                  -- ¡SIEMPRE con WHERE!

-- ── DML: eliminar ──────────────────────────────────────────
DELETE FROM tabla
WHERE condición;                  -- ¡SIEMPRE con WHERE!

-- ── Metainformación ────────────────────────────────────────
SELECT name FROM sqlite_master WHERE type='table';
PRAGMA foreign_keys = ON;
```

## ¿Qué viene en la próxima clase?

En la **Clase 2** aprendemos a **consultar datos** con `SELECT`, `WHERE`, `ORDER BY`, `LIMIT` y las funciones de agregación (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`).

In [ ]:
conn.close()
print("Conexión cerrada")